In [1]:
#remove when converting to .py file
from pathlib import Path
import importlib.util
import numpy as np
import librosa
import torch
from transformers import pipeline, ClapProcessor, ClapModel
import json

print(np.isnan(0.0))
PROJECT_ROOT = Path.cwd()
helper_path = PROJECT_ROOT / ".." /"src" / "utils.py"
spec = importlib.util.spec_from_file_location("utils", helper_path)
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)

/home/niagui/miniconda3/envs/transformers/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-18 16:04:39.211221: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-18 16:04:39.276799: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-18 16:04:40.589534: I tensorflow/core/util/port.cc:153] oneDNN custom operatio

ImportError: numpy.core.multiarray failed to import (auto-generated because you didn't call 'numpy.import_array()' after cimporting numpy; use '<void>numpy._import_array' to disable if you are certain you don't need it).

In [ ]:
torch.backends.cudnn.enabled = True
if torch.cuda.is_available():
    print("GPU(s):", torch.cuda.device_count(), torch.cuda.get_device_name(0))  #you need cuda otherwise set device to cpu

GPU(s): 1 NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
audio_classifier = pipeline(task="zero-shot-audio-classification", model="laion/larger_clap_general", batch=8, device='cuda')   #SET IT HERE

Device set to use cuda


In [ ]:
model = ClapModel.from_pretrained("laion/larger_clap_general")
processor = ClapProcessor.from_pretrained("laion/larger_clap_general")

In [ ]:
audio_segments_path = '../segments/testSong'
audio = "../audio/testSong.mp3"

In [ ]:
clap_label_json = "../json/clap_labels.json"
with open(clap_label_json, 'r') as f:
    music_labels = json.load(f)

anchor_labels_json = "../json/anchor_labels.json"
with open(anchor_labels_json, 'r') as f:
    anchor_labels = json.load(f)

time_segments_json = "../json/k_beat_segments.json"
with open(time_segments_json, 'r') as f:
    k_beats_segments = json.load(f)

segmentation_json = "../json/segmentation.json"
with open(segmentation_json, 'r') as f:
    s = json.load(f)
    major_seg_times, major_seg_sim = s[11][0], s[11][1]



In [ ]:
def clap_classification(audio, time_base, labels, sr=22050, k=3, threshold=0.1):
    result = []
    y, sr = librosa.load(audio, sr=sr)
    for [start, end] in time_base:
        chunk = y[int(round(start * sr)): int(round(end * sr))]     #select chunk
        features = {}
        for label in labels:
            classes = labels[label]
            predictions = audio_classifier(chunk, candidate_labels = classes)

            top_preds = sorted(predictions, key=lambda x: x['score'], reverse=True)
            filtered = top_preds[:k]
            features[label] = filtered

        result.append({
            "start": start,
            "end": end,
            "feature": features
        })

    return result

In [ ]:
clap_result = clap_classification(audio, k_beats_segments, labels=music_labels, sr=22050, k=3, threshold=0.1)
utils.save_as_json("clap_results", clap_result)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
major_seg_clap_res = clap_classification(audio, major_seg_times, labels=music_labels, sr=22050, k=3, threshold=0.1)
utils.save_as_json("major_seg_clap_res", major_seg_clap_res)

In [ ]:
len(clap_result)

61

In [ ]:
def get_text_embedding(text:list):
    """
        returns a pyTorch array.
    """
    with torch.no_grad():
        inputs = processor(text=text, return_tensors="pt", padding=True, truncation=True)
        emb = model.get_text_features(**inputs)
        emb = torch.nn.functional.normalize(emb, dim=-1)
    return emb

In [ ]:
emb = get_text_embedding(anchor_labels)
print(emb)
print(emb[0])

tensor([[-0.0117, -0.0809, -0.0598,  ..., -0.0225,  0.0447,  0.0254],
        [-0.0603, -0.0418, -0.0608,  ..., -0.0495,  0.0014,  0.0116],
        [-0.0854, -0.0308, -0.0664,  ..., -0.0188, -0.0082,  0.0342],
        ...,
        [-0.0762, -0.0485, -0.0683,  ..., -0.0432,  0.0272,  0.0216],
        [-0.0149, -0.0121, -0.1063,  ...,  0.0181,  0.0413,  0.0261],
        [-0.0225, -0.0713, -0.0597,  ..., -0.0220,  0.0517,  0.0351]])
tensor([-0.0117, -0.0809, -0.0598,  0.0670,  0.0153,  0.0752,  0.0045,  0.0490,
        -0.0559,  0.0120, -0.0641, -0.0499, -0.0061, -0.0361, -0.0586,  0.0136,
        -0.0620,  0.0566,  0.0681, -0.0459,  0.0249, -0.0197, -0.0191, -0.0317,
        -0.0626,  0.0459,  0.0284,  0.0416, -0.0122,  0.0116, -0.0536, -0.0012,
         0.0071, -0.0071,  0.0574, -0.0126,  0.0299,  0.0099,  0.0770,  0.0319,
         0.0311,  0.0462,  0.0063,  0.0135,  0.0249, -0.0712, -0.0119, -0.0106,
         0.0021, -0.0661, -0.0278, -0.0161, -0.0859,  0.0913,  0.0096,  0.0460,
      

In [ ]:
anchor_labels_emb = {}
for i, label in enumerate(anchor_labels):
    anchor_labels_emb[label] = emb[i].tolist()

utils.save_as_json("anchor_labels_embed", anchor_labels_emb)


In [ ]:
# find consine simularity between anchor labels and new label

v1 = np.array(anchor_labels_emb["happy"])   #replace this with the llm generated ones
v2 = np.array(get_text_embedding("whimsical").tolist()[0])

consine_sim = v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2))
print(consine_sim)

0.8267642263506079


In [ ]:
def get_cosine_simularity(new_label, anchor_labels, center=True, remove_pcs=0):
    new = np.array(new_label)
    anchor = np.array(anchor_labels)

    new = new / np.linalg.norm(new)
    anchor = anchor / np.linalg.norm(anchor, axis=1, keepdims=True)


    if center:
        mu = anchor.mean(axis=0, keepdims=True)
        anchor = anchor - mu
        new = new - mu.squeeze(0)

    ##idk what this is but chat said it can help remove "directionness"
    if remove_pcs and remove_pcs > 0:
        # SVD computes principal components
        U, S, VT = np.linalg.svd(anchor, full_matrices=False)
        P = VT[:remove_pcs].T  # (d, k)

        # Project out dominant PCs
        anchor = anchor - (anchor @ P) @ P.T
        new = new - (new @ P) @ P.T
    
    similarities = anchor @ new
    return similarities

In [ ]:
val = list(anchor_labels_emb.values())
key = list(anchor_labels_emb.keys())
sims = get_cosine_simularity(v2, val)
z = zip(sims, key)
for i, j in z:
    print(i,j)

0.03386036860748084 happy
-0.01244910650411513 sad
-0.06310178316916043 sleepy
0.050613317407713174 brave
-0.04108050434384675 grumpy
-0.03569988522593303 scared
0.06785759322786136 shy


In [ ]:
top_k = 3   #pick top k anchors to compare with
temperature = 0.05  #controls how sharp on the weighting

idx = np.argpartition(-sims, top_k)[:top_k]
print(idx)
w = np.zeros_like(sims)
w[idx] = np.exp(sims[idx] / temperature)
w /= (w.sum() + np.finfo(float).eps)
print(w)


[6 3 0]
[0.22873921 0.         0.         0.31978263 0.         0.
 0.45147816]
